Do not Run All.

# Manga Task: Qwen3.5-4B QLoRA fine-tuning

This notebook adapts the cited Unsloth Qwen3.5 vision notebook for the three-page manga dataset. It loads the project configuration, checks the GPU, trains a 4-bit QLoRA adapter, and evaluates validation predictions.
Run GPU cells manually and inspect their output before continuing.

**Do not Run All. Run cell by cell and read the output after each GPU cell.**

## Table of contents

0. Setup and environment check  
1. Configuration (single source of truth)  
2. Data: loading sequences and building samples  
3. Token and length report  
4. Load the model (4-bit QLoRA)  
5. One-sample forward pass (sanity check)  
6. Fit test: choose max_pixels for 8GB VRAM  
7. Baseline: zero-shot validation (no adapter)  
8. Training  
9. Validation inference with the trained adapter  
10. Evaluation and results table  
11. Save, reload, and reproducibility notes

## 0. Setup and environment check

This CPU-side check confirms that the notebook is using the intended Python environment and that CUDA is available before loading a model. GPU cell: run manually.
What to check in the output: Python 3.12, `torch.cuda.is_available() is True`, the RTX 4060 name, bf16 support, and package versions.

In [ ]:
import importlib.util
import platform
import sys

print('Python:', sys.version)
if importlib.util.find_spec('torch') is None:
    raise RuntimeError('torch is required on the training machine; install it in the terminal before running this notebook.')
import torch
if not torch.cuda.is_available():
    raise RuntimeError('CUDA is required; torch.cuda.is_available() is False.')
gpu = torch.cuda.get_device_properties(0)
print('Platform:', platform.platform())
print('CUDA:', torch.version.cuda)
print('GPU:', gpu.name)
print('bf16 supported:', torch.cuda.is_bf16_supported())
for package in ['unsloth', 'transformers', 'trl', 'tokenizers', 'peft', 'bitsandbytes']:
    module = __import__(package)
    print(f'{package}:', getattr(module, '__version__', 'version unavailable'))

## 1. Configuration (single source of truth)

This cell loads all model, training, output, and experiment settings from `configs/` through `src/config.py`. The values in `configs/base.yaml` are starting points, not tuned results; change them there, not in later cells. CPU-only setup.
What to check in the output: the model ID, experiment name, split sizes, and the displayed `max_pixels`, `max_length`, learning rate, LoRA values, and seed.

In [ ]:
from pathlib import Path
from datetime import datetime
import json
import importlib
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'configs' / 'base.yaml').exists():
    PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
importlib.invalidate_caches()
sys.modules.pop('config', None)
from config import load_config, save_resolved_config, set_seed
from prompts import build_messages

config = load_config(PROJECT_ROOT, profile='cuda_4060')
set_seed(config.random_seed)
experiment = f'{config.experiment_name}_{datetime.now().strftime("%Y%m%d_%H%M%S")}'
result_dir = PROJECT_ROOT / config.output_dir / experiment
adapter_dir = PROJECT_ROOT / config.adapter_output_dir / experiment
log_dir = PROJECT_ROOT / 'logs' / experiment
result_dir.mkdir(parents=True, exist_ok=True)
adapter_dir.mkdir(parents=True, exist_ok=True)
log_dir.mkdir(parents=True, exist_ok=True)
save_resolved_config(config, result_dir)
print({
    'model': config.hf_model_name,
    'revision': config.hf_model_revision,
    'max_pixels': config.max_pixels,
    'max_length': config.max_length,
    'epochs': config.num_epochs,
    'learning_rate': config.learning_rate,
    'lora_r': config.lora_r,
    'lora_alpha': config.lora_alpha,
    'seed': config.random_seed,
    'experiment': experiment,
})

## 2. Data: loading sequences and building samples

The project loader keeps all three pages together and builds messages with the three images before the shared prompt. It uses the training split only for fitting and keeps label text unchanged. CPU-only setup.
What to check in the output: train and validation sequence counts, and that the paths and target data load without schema errors.

In [ ]:
from prepare import load_examples

split = json.loads((PROJECT_ROOT / config.split_path).read_text(encoding='utf-8'))
train_examples = list(load_examples(PROJECT_ROOT / config.dataset_path, split['train']))
val_examples = list(load_examples(PROJECT_ROOT / config.dataset_path, split['val']))
print({
    'train_sequences': len(train_examples),
    'val_sequences': len(val_examples),
    'sample_image_paths': train_examples[0]['image_paths'],
    'sample_target_json_length': len(train_examples[0]['target_json']),
})

## 3. Token and length report

This CPU-only cell uses the cached official processor and PIL images to count image tokens, target tokens, and total tokens for every training sample. The verified mechanism is `images_kwargs.size.longest_edge`; it also forces `enable_thinking=False`.
What to check in the output: counts change across native, 700k, and 500k; the 700k p99 plus 5% agrees with `max_length=3152`; the target p99 plus margin agrees with `max_new_tokens=1008`; and the no-truncation assertion passes.

In [ ]:
import os
from io import BytesIO
from PIL import Image
from transformers import AutoProcessor
if 'train_examples' not in globals():
    from prepare import load_examples
    _split = json.loads((PROJECT_ROOT / config.split_path).read_text(encoding='utf-8'))
    train_examples = list(load_examples(PROJECT_ROOT / config.dataset_path, _split['train']))
    print('Loaded train_examples for token report:', len(train_examples))

os.environ['HF_HUB_OFFLINE'] = '1'
MODEL_REPO = config.hf_model_name
MODEL_REVISION = config.hf_model_revision
HF_CACHE = Path(os.environ.get('HF_HUB_CACHE', Path.home() / '.cache' / 'huggingface' / 'hub'))
PROCESSOR_CACHE = HF_CACHE / f'models--{MODEL_REPO.replace(chr(47), chr(45) + chr(45))}' / 'snapshots' / MODEL_REVISION
if not PROCESSOR_CACHE.exists():
    raise FileNotFoundError(f'Cached processor files not found: {PROCESSOR_CACHE}')
processor = AutoProcessor.from_pretrained(PROCESSOR_CACHE, revision=MODEL_REVISION, local_files_only=True)
native_max_pixels = int(processor.image_processor.size['longest_edge'])
candidates = [native_max_pixels, 700_000, 500_000]

class ThinkingOffProcessor:
    def __init__(self, processor):
        self._processor = processor
    def apply_chat_template(self, messages, **kwargs):
        kwargs['enable_thinking'] = False
        return self._processor.apply_chat_template(messages, **kwargs)
    def __call__(self, *args, **kwargs):
        return self._processor(*args, **kwargs)
    def __getattr__(self, name):
        return getattr(self._processor, name)

def _as_pil_image(value):
    if isinstance(value, Image.Image):
        return value
    if isinstance(value, dict) and value.get('bytes'):
        return Image.open(BytesIO(value['bytes'])).convert('RGB')
    if isinstance(value, dict) and value.get('path'):
        return Image.open(value['path']).convert('RGB')
    raise TypeError(f'Unsupported image value from dataset: {type(value)}')

class ResponseOnlyVisionDataCollator:
    def __init__(self, model, processor, max_seq_length):
        self.processor = processor
        self.base = UnslothVisionDataCollator(model, processor, max_seq_length=max_seq_length, resize='max')
    def __call__(self, examples):
        batch = self.base(examples)
        for row, example in enumerate(examples):
            prompt_messages = example['messages'][:-1]
            prompt_text = self.processor.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
            images = [_as_pil_image(part['image']) for message in prompt_messages for part in message['content'] if part.get('type') == 'image']
            prompt_batch = self.processor(text=prompt_text, images=images, return_tensors='pt', add_special_tokens=False)
            prefix_length = int(prompt_batch['input_ids'].shape[-1])
            if prefix_length > batch['labels'].shape[1]:
                raise ValueError('The prompt was truncated before the target JSON.')
            batch['labels'][row, :prefix_length] = -100
            end_token_id = self.processor.tokenizer.convert_tokens_to_ids('<|im_end|>')
            end_positions = torch.where(batch['input_ids'][row, prefix_length:] == end_token_id)[0]
            if len(end_positions) == 0:
                raise ValueError('The target JSON closing <|im_end|> token was not found.')
            batch['labels'][row, prefix_length + int(end_positions[0]) + 1:] = -100
        return batch

def pil_messages(example, max_pixels=None):
    messages = build_messages(example['image_paths'], example['target_json'])
    converted = []
    for message in messages:
        content = []
        for item in message['content']:
            if item['type'] == 'image':
                image_item = {'type': 'image', 'image': Image.open(item['image']).convert('RGB')}
                if max_pixels is not None:
                    image_item['max_pixels'] = max_pixels
                content.append(image_item)
            else:
                content.append(item)
        converted.append({'role': message['role'], 'content': content})
    return converted

def open_images(image_paths):
    return [Image.open(path).convert('RGB') for path in image_paths]

def encode_cpu(example, max_pixels):
    return processor.apply_chat_template(
        pil_messages(example), tokenize=True, return_dict=True, return_tensors='pt',
        add_generation_prompt=False, enable_thinking=False,
        images_kwargs={'size': {'shortest_edge': 65_536, 'longest_edge': max_pixels}},
    )

def nearest_rank(values, fraction):
    ordered = sorted(values)
    return ordered[min(len(ordered) - 1, max(0, math.ceil(fraction * len(ordered)) - 1))]

import gc, math, statistics, time
report = {}
for max_pixels in candidates:
    totals, targets, image_tokens = [], [], []
    for example in train_examples:
        encoded = encode_cpu(example, max_pixels)
        grid = encoded['image_grid_thw']
        merge = int(processor.image_processor.merge_size)
        image_tokens.extend(int(row[1] * row[2] // (merge * merge)) for row in grid)
        totals.append(int(encoded['input_ids'].shape[-1]))
        targets.append(int(processor.tokenizer(example['target_json'], add_special_tokens=False, return_tensors='pt')['input_ids'].shape[-1]))
        del encoded, grid
        gc.collect()
    report[max_pixels] = {'totals': totals, 'targets': targets, 'image_tokens': image_tokens}
    print(max_pixels, 'image tokens/page min/median/p95/max', min(image_tokens), statistics.median(image_tokens), nearest_rank(image_tokens, .95), max(image_tokens), 'total min/median/p95/max', min(totals), statistics.median(totals), nearest_rank(totals, .95), max(totals), 'over 4k/6k/8k', [sum(value > limit for value in totals) / len(totals) for limit in (4000, 6000, 8000)])

chosen = report[700_000]
measured_max_length = math.ceil(nearest_rank(chosen['totals'], .99) * 1.05)
measured_max_new_tokens = math.ceil(nearest_rank(chosen['targets'], .99) * 1.05)
if config.max_length != measured_max_length:
    raise AssertionError(f'Config max_length={config.max_length}; measured required value={measured_max_length}. Update configs/base.yaml.')
if config.max_new_tokens != measured_max_new_tokens:
    raise AssertionError(f'Config max_new_tokens={config.max_new_tokens}; measured required value={measured_max_new_tokens}. Update configs/base.yaml.')
too_long = [example['sequence_id'] for example, value in zip(train_examples, chosen['totals']) if value > config.max_length]
assert not too_long, f'Training samples exceed max_length: {too_long}'
print('target JSON median/p95/max:', statistics.median(chosen['targets']), nearest_rank(chosen['targets'], .95), max(chosen['targets']))
print('max_length:', config.max_length, 'max_new_tokens:', config.max_new_tokens)
print('max_pixels mechanism verified: images_kwargs.size.longest_edge; first sample counts:', {px: report[px]['image_tokens'][:3] for px in candidates})
dry_batch = encode_cpu(train_examples[0], 700_000)
decoded = processor.tokenizer.decode(dry_batch['input_ids'][0], skip_special_tokens=False)
for label in ['character1', 'character2', 'character3', 'character4', 'character5', 'character6', 'character7', 'character8', 'UNKNOWN', 'NARRATION']:
    decoded = decoded.replace(label, f'[{label}]')
print('CPU dry-run shapes:', {key: tuple(value.shape) for key, value in dry_batch.items() if hasattr(value, 'shape')})
print('decoded text with speaker labels marked:', decoded[:2000])

## 4. Load the model (4-bit QLoRA)

This GPU cell loads the cached local model in 4-bit mode and refuses network access with `local_files_only=True`. It sets the verified processor size mechanism and uses Unsloth gradient checkpointing. GPU cell: run manually.
What to check in the output: bf16 support is `True`, no download attempt occurs, `max_pixels` equals the config, and VRAM after load is reported.

In [ ]:
import os
import torch
from unsloth import FastVisionModel
os.environ['HF_HUB_OFFLINE'] = '1'

if not torch.cuda.is_available() or not torch.cuda.is_bf16_supported():
    raise RuntimeError('This RTX 4060 run requires CUDA and bf16 support.')
if config.load_in_4bit is not True:
    raise RuntimeError('This notebook requires config.load_in_4bit=True.')
model, tokenizer = FastVisionModel.from_pretrained(
    model_name=config.hf_model_name, revision=config.hf_model_revision, local_files_only=True,
    load_in_4bit=True, max_seq_length=config.max_length,
    use_gradient_checkpointing='unsloth',
)
image_processor = getattr(tokenizer, 'image_processor', None)
if image_processor is None or not hasattr(image_processor, 'size'):
    raise RuntimeError('The cached processor has no verified image size interface.')
image_processor.size['longest_edge'] = config.max_pixels
assert image_processor.size['longest_edge'] == config.max_pixels
print('bf16 supported:', torch.cuda.is_bf16_supported())
print('explicit image size:', image_processor.size)
print('VRAM reserved after load (GB):', round(torch.cuda.memory_reserved() / 1024**3, 3))

## 5. One-sample forward pass (sanity check)

This cell sends one validation sequence through the base model to check image loading, prompt formatting, and greedy decoding before training. GPU cell: run manually.
What to check in the output: generation completes without a CUDA or image-processing error and returns text that can be inspected; this is a smoke test, not a result claim.

In [ ]:
from transformers import TextStreamer

FastVisionModel.for_inference(model)
one = val_examples[0]
messages = build_messages(one['image_paths'])
prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
inputs = tokenizer(images=open_images(one['image_paths']), text=prompt_text, return_tensors='pt', truncation=False).to('cuda')
with torch.inference_mode():
    output_ids = model.generate(**inputs, do_sample=False, max_new_tokens=config.max_new_tokens, use_cache=True)
print(tokenizer.decode(output_ids[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True))

## 6. Fit test: choose max_pixels for 8GB VRAM

This optional GPU cell runs five training steps on the first eight training samples and can loop over the three measured resolutions. It loads each candidate locally, catches out-of-memory errors, and frees memory between runs. GPU cell: run manually.
What to check in the output: loss values are printed, seconds per step is finite, and peak reserved VRAM stays below about 7 GB; an OOM message names the candidate that failed.

In [ ]:
# CPU-only collator smoke test: this uses config JSON, not model weights.
import torch
from types import SimpleNamespace
from transformers import AutoConfig
from datasets import Dataset
from unsloth.trainer import UnslothVisionDataCollator

processor.image_processor.size['longest_edge'] = 700_000
smoke_config = AutoConfig.from_pretrained(PROCESSOR_CACHE, local_files_only=True)
smoke_model = SimpleNamespace(config=smoke_config, max_seq_length=config.max_length, get_input_embeddings=lambda: SimpleNamespace(weight=torch.empty(1)))
smoke_rows = Dataset.from_list([{'messages': pil_messages(example, 700_000)} for example in train_examples[:2]])
smoke_examples = [smoke_rows[index] for index in range(2)]
smoke_processor = ThinkingOffProcessor(processor)
smoke_collator = ResponseOnlyVisionDataCollator(smoke_model, smoke_processor, config.max_length)
smoke_batch = smoke_collator(smoke_examples)
assert smoke_batch['input_ids'].shape == smoke_batch['labels'].shape
first_sample_image_tokens = [int(grid[1] * grid[2] // (processor.image_processor.merge_size ** 2)) for grid in smoke_batch['image_grid_thw'][:3]]
assert first_sample_image_tokens == [651, 651, 672], first_sample_image_tokens
print('CPU collator input_ids shape:', tuple(smoke_batch['input_ids'].shape))
print('CPU collator pixel_values shape:', tuple(smoke_batch['pixel_values'].shape))
print('CPU collator image_grid_thw:', smoke_batch['image_grid_thw'].tolist())
for row, example in enumerate(smoke_examples):
    labelled = torch.where(smoke_batch['labels'][row] != -100)[0]
    target_ids = processor.tokenizer(example['messages'][-1]['content'][0]['text'], add_special_tokens=False, return_tensors='pt')['input_ids'][0]
    assert smoke_batch['labels'].shape[1] == smoke_batch['input_ids'].shape[1]
    assert int(smoke_batch['input_ids'][row, labelled[0]]) == int(target_ids[0]), (row, int(smoke_batch['input_ids'][row, labelled[0]]), int(target_ids[0]))
    assert processor.tokenizer.decode([int(smoke_batch['labels'][row, labelled[-1]])], skip_special_tokens=False) == '<|im_end|>'
    print({'row': row, 'input_ids_len': int(smoke_batch['input_ids'][row].numel()), 'labels_len': int(smoke_batch['labels'][row].numel()), 'labelled_count': int(labelled.numel()), 'first_label_is_target_start': True, 'last_label': '<|im_end|>'})
print('first sample image tokens:', first_sample_image_tokens)
print('CPU collator smoke test: PASS')

In [ ]:
import gc, time
from datasets import Dataset
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTConfig, SFTTrainer

fit_results = []
def run_fit_test(max_pixels, steps=5, sample_count=8):
    fit_model = None
    try:
        fit_model, fit_tokenizer = FastVisionModel.from_pretrained(
            model_name=config.hf_model_name, revision=config.hf_model_revision, local_files_only=True, load_in_4bit=True,
            max_seq_length=config.max_length, use_gradient_checkpointing='unsloth',
        )
        fit_tokenizer.image_processor.size['longest_edge'] = max_pixels
        assert fit_tokenizer.image_processor.size['longest_edge'] == max_pixels
        fit_model = FastVisionModel.get_peft_model(fit_model, finetune_vision_layers=False, finetune_language_layers=True, r=config.lora_r, lora_alpha=config.lora_alpha, lora_dropout=config.lora_dropout, random_state=config.random_seed)
        fit_dataset = Dataset.from_list([{'messages': pil_messages(x, max_pixels)} for x in train_examples[:sample_count]])
        fit_args = SFTConfig(output_dir=str(result_dir / f'fit_test_{max_pixels}'), max_steps=steps, per_device_train_batch_size=1, gradient_accumulation_steps=1, learning_rate=config.learning_rate, optim='adamw_8bit', logging_steps=1, report_to='none', bf16=True, remove_unused_columns=False, dataset_text_field='', dataset_kwargs={'skip_prepare_dataset': True}, max_length=config.max_length)
        fit_processor = ThinkingOffProcessor(fit_tokenizer)
        fit_trainer = SFTTrainer(model=fit_model, tokenizer=fit_tokenizer, data_collator=ResponseOnlyVisionDataCollator(fit_model, fit_processor, config.max_length), train_dataset=fit_dataset, args=fit_args)
        torch.cuda.reset_peak_memory_stats()
        started = time.perf_counter(); stats = fit_trainer.train(); elapsed = time.perf_counter() - started
        losses = [x.get('loss') for x in fit_trainer.state.log_history if x.get('loss') is not None]
        result = {'max_pixels': max_pixels, 'status': 'ok', 'losses': losses, 'seconds_per_step': elapsed / steps, 'peak_reserved_gb': torch.cuda.max_memory_reserved() / 1024**3, 'peak_allocated_gb': torch.cuda.max_memory_allocated() / 1024**3}
        fit_results.append(result)
        print(result)
    except Exception as error:
        result = {'max_pixels': max_pixels, 'status': 'error', 'exception_type': type(error).__name__, 'message': str(error)}
        fit_results.append(result)
        print(result)
    finally:
        del fit_model
        gc.collect(); torch.cuda.empty_cache()

run_native_fit_test = False  # Native resolution reaches about 5,311 tokens; opt in only deliberately.
fit_candidates = [700_000, 500_000] + ([native_max_pixels] if run_native_fit_test else [])
for candidate in fit_candidates:
    run_fit_test(candidate)

## 7. Baseline: zero-shot validation (no adapter)

This cell runs the unadapted model on the validation split using the same message builder and parser used later. GPU cell: run manually.
What to check in the output: the number of processed sequences equals the validation split, parse/schema failures are `0` if the model follows the format, and the JSONL path is under the experiment results directory.

In [ ]:
from postprocess import parse_prediction_text, validate_prediction

def predict_examples(examples, output_path, inference_model, inference_tokenizer, tag):
    rows = []
    parse_failures = 0
    for example in examples:
        messages = build_messages(example['image_paths'])
        prompt_text = inference_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
        inputs = inference_tokenizer(images=open_images(example['image_paths']), text=prompt_text, return_tensors='pt', truncation=False).to('cuda')
        with torch.inference_mode():
            generated = inference_model.generate(**inputs, do_sample=False, max_new_tokens=config.max_new_tokens, use_cache=True)
        raw = inference_tokenizer.decode(generated[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True)
        try:
            parsed = parse_prediction_text(raw)
            row = {'sequence_id': example['sequence_id'], 'pages': parsed['pages']}
            issues = validate_prediction(row)
            if issues:
                raise ValueError('; '.join(issues))
        except (ValueError, TypeError, json.JSONDecodeError):
            parse_failures += 1
            row = {'sequence_id': example['sequence_id'], 'pages': [[], [], []]}
        rows.append(row)
    output_path.write_text(''.join(json.dumps(row, ensure_ascii=False, separators=(',', ':')) + '\n' for row in rows), encoding='utf-8')
    print('tag:', tag, 'sequences:', len(rows), 'parse/schema failures:', parse_failures)
    return rows

baseline_path = result_dir / 'val_predictions_baseline.jsonl'
baseline_rows = predict_examples(val_examples, baseline_path, model, tokenizer, 'baseline')
print('Baseline predictions:', baseline_path)

## 8. Training

These cells record memory before training, train the LoRA adapter, and record final memory and runtime. GPU cells: run manually and pause after each one.
What to check in the output: batch size `1`, gradient accumulation `8`, the configured epoch and learning rate, TensorBoard logging, and a peak VRAM value that fits the 8 GB card. The config values are starting points, not tuned results.

### 8.1 Memory before training

This preserves the reference notebook's memory-statistics check. What to check: the expected RTX 4060 name and the initial reserved memory.

In [ ]:
# Adapted from the reference notebook's current memory-statistics cell.
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f'GPU = {gpu_stats.name}. Max memory = {max_memory} GB.')
print(f'{start_gpu_memory} GB of memory reserved.')

### 8.2 Build the trainer and train

This cell creates the LoRA adapter and trains only on the prepared training split. It checks that the installed TRL exposes assistant-only loss before starting. GPU cell: run manually.
What to check in the output: no assistant-loss capability error, no truncation error, a completed training summary, and the adapter saved under `weights/<experiment>/`.

In [ ]:
import inspect
import gc
# Release objects left by baseline/fit-test cells before allocating training state.
for _stale_name in ('trainer', 'trainer_stats', 'data_collator', 'training_processor', 'inputs', 'generated', 'output_ids', 'raw'):
    globals().pop(_stale_name, None)
gc.collect()
torch.cuda.empty_cache()
_free_bytes, _total_bytes = torch.cuda.mem_get_info()
print('GPU free before trainer setup (GB):', round(_free_bytes / 1024**3, 3))
from torch.utils.data import Dataset as TorchDataset
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTConfig, SFTTrainer

FastVisionModel.for_training(model)
if not hasattr(model, 'peft_config'):
    model = FastVisionModel.get_peft_model(
        model,
        finetune_vision_layers=config.finetune_vision_layers,
        finetune_language_layers=True,
        finetune_attention_modules=True,
        finetune_mlp_modules=True,
        r=config.lora_r,
        lora_alpha=config.lora_alpha,
        lora_dropout=config.lora_dropout,
        bias='none',
        random_state=config.random_seed,
    )
else:
    print('LoRA adapters already present; reusing the existing adapted model.')
class LazyMangaDataset(TorchDataset):
    def __init__(self, examples):
        self.examples = examples
    def __len__(self):
        return len(self.examples)
    def __getitem__(self, index):
        example = self.examples[index]
        return {'messages': pil_messages(example, config.max_pixels)}

train_dataset = LazyMangaDataset(train_examples)
training_processor = ThinkingOffProcessor(tokenizer)
data_collator = ResponseOnlyVisionDataCollator(model, training_processor, config.max_length)
collator_signature = inspect.signature(UnslothVisionDataCollator.__init__)
sft_parameters = inspect.signature(SFTConfig).parameters
print('thinking disabled in collator:', True, 'response-only label masking:', True, 'collator args:', list(collator_signature.parameters))

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=data_collator,
    train_dataset=train_dataset,
    args=SFTConfig(
        per_device_train_batch_size=config.per_device_batch,
        gradient_accumulation_steps=config.grad_accum,
        num_train_epochs=config.num_epochs,
        learning_rate=config.learning_rate,
        optim='adamw_8bit',
        warmup_ratio=0.1,
        lr_scheduler_type='cosine',
        weight_decay=0.001,
        seed=config.random_seed,
        output_dir=str(result_dir),
        logging_dir=str(log_dir),
        report_to='tensorboard',
        dataloader_num_workers=0,
        dataset_num_proc=1,
        remove_unused_columns=False,
        dataset_text_field='',
        dataset_kwargs={'skip_prepare_dataset': True},
        max_length=config.max_length,
        bf16=True,
        save_strategy='epoch',
    ),
)
trainer_stats = trainer.train()
trainer.save_state()
(result_dir / 'trainer_log_history.json').write_text(json.dumps(trainer.state.log_history, indent=2), encoding='utf-8')
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print('Adapter saved:', adapter_dir)

### 8.3 Memory after training

This preserves the reference notebook's final memory and runtime report. GPU cell: run manually after training finishes.
What to check in the output: training runtime, peak reserved memory, and peak memory percentage; record an out-of-memory failure rather than changing settings silently.

In [ ]:
# Adapted from the reference notebook's final memory-statistics cell.
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f'Peak reserved memory = {used_memory} GB.')
print(f'Peak reserved memory for training = {used_memory_for_lora} GB.')
print(f'Peak reserved memory % of max memory = {used_percentage} %.')
print(f'Peak reserved memory for training % of max memory = {lora_percentage} %.')

## 9. Validation inference with the trained adapter

This cell reuses the greedy inference and defensive parser after training, writing one JSON object per validation sequence. GPU cell: run manually.
What to check in the output: the validation count is correct, every output has three page lists, parse failures are `0` when possible, and the JSONL file is saved under the experiment results directory.

### 9.1 Adapter reload check

This GPU cell reloads the cached 4-bit base plus the saved adapter and runs one validation sequence. It uses the same greedy function and disables thinking with `enable_thinking=False`.
What to check in the output: the adapter loads locally, one JSONL row is written, and the parser reports zero failures for that row.

In [9]:
from peft import PeftModel
from postprocess import parse_prediction_text, validate_prediction

if 'predict_examples' not in globals():
    def predict_examples(examples, output_path, inference_model, inference_tokenizer, tag):
        rows = []
        parse_failures = 0
        for example in examples:
            messages = build_messages(example['image_paths'])
            prompt_text = inference_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
            inputs = inference_tokenizer(images=open_images(example['image_paths']), text=prompt_text, return_tensors='pt', truncation=False).to('cuda')
            with torch.inference_mode():
                generated = inference_model.generate(**inputs, do_sample=False, max_new_tokens=config.max_new_tokens, use_cache=True)
            raw = inference_tokenizer.decode(generated[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True)
            try:
                parsed = parse_prediction_text(raw)
                row = {'sequence_id': example['sequence_id'], 'pages': parsed['pages']}
                issues = validate_prediction(row)
                if issues:
                    raise ValueError('; '.join(issues))
            except (ValueError, TypeError, json.JSONDecodeError):
                parse_failures += 1
                row = {'sequence_id': example['sequence_id'], 'pages': [[], [], []]}
            rows.append(row)
        output_path.write_text(''.join(json.dumps(row, ensure_ascii=False, separators=(',', ':')) + '\n' for row in rows), encoding='utf-8')
        print('tag:', tag, 'sequences:', len(rows), 'parse/schema failures:', parse_failures)
        return rows

if 'reload_model' not in globals():
    reload_base, reload_tokenizer = FastVisionModel.from_pretrained(
        model_name=config.hf_model_name, revision=config.hf_model_revision, local_files_only=True, load_in_4bit=True,
        max_seq_length=config.max_length, use_gradient_checkpointing='unsloth',
    )
    reload_model = PeftModel.from_pretrained(reload_base, str(adapter_dir), local_files_only=True)
    reload_tokenizer.image_processor.size['longest_edge'] = config.max_pixels
    reload_model.eval()
else:
    print('Adapter reload already present; reusing it.')
reload_check_path = result_dir / 'adapter_reload_check.jsonl'
predict_examples(val_examples[:1], reload_check_path, reload_model, reload_tokenizer, 'adapter_reload_check')
model, tokenizer = reload_model, reload_tokenizer

val_path = result_dir / 'val_predictions_finetuned.jsonl'
val_rows = predict_examples(val_examples, val_path, model, tokenizer, 'finetuned')
print('Fine-tuned validation predictions:', val_path)

Adapter reload already present; reusing it.
tag: adapter_reload_check sequences: 1 parse/schema failures: 0
tag: finetuned sequences: 16 parse/schema failures: 0
Fine-tuned validation predictions: d:\byte task2 manga adv\results\qwen35_4b_20261010_181416\val_predictions_finetuned.jsonl


## 10. Evaluation and results table

This cell calls the project's official scorer wrapper, which writes filtered references and scores under the experiment directory. CPU orchestration, but it invokes the scorer after GPU inference.
What to check in the output: the scores file exists, all requested metrics are present, and missing or extra sequence IDs are not reported. No result claim is made until this cell has run.

In [10]:
import subprocess

eval_command = [
    sys.executable, str(PROJECT_ROOT / 'src' / 'eval.py'), str(val_path),
    '--experiment', experiment, '--profile', 'cuda_4060',
]
subprocess.run(eval_command, cwd=PROJECT_ROOT, check=True)
scores_path = result_dir / 'scores.json'
scores = json.loads(scores_path.read_text(encoding='utf-8'))
print(json.dumps(scores, indent=2))
import csv
experiments_path = PROJECT_ROOT / config.output_dir / 'experiments.csv'
row = {'experiment': experiment, 'max_pixels': config.max_pixels, 'max_length': config.max_length, 'epochs': config.num_epochs, **scores.get('macro', {})}
write_header = not experiments_path.exists()
with experiments_path.open('a', newline='', encoding='utf-8') as handle:
    writer = csv.DictWriter(handle, fieldnames=list(row))
    if write_header: writer.writeheader()
    writer.writerow(row)
print('Results row appended:', experiments_path)

{
  "version": "2.1",
  "pages_per_sequence": 3,
  "macro": {
    "text_order_score": 0.7292300694972508,
    "balanced_joint_f1": 0.41166653389716246,
    "speaker_accuracy_on_matched": 0.6424331986691283,
    "joint_f1": 0.5172795215594119,
    "matched_token_coverage": 0.8027727774464416
  },
  "missing_sequences": [],
  "sequences": {
    "seq_032cbb4dc8686963": {
      "text_order_score": 0.8682279978534492,
      "active_text_pages": 3,
      "balanced_joint_f1": 0.8477681057475307,
      "balanced_speaker_mapping": {
        "NARRATION": "NARRATION",
        "character1": "character1",
        "character2": "character2"
      },
      "speaker_accuracy_on_matched": 0.9240506329113924,
      "joint_f1": 0.8044077134986226,
      "matched_token_coverage": 0.8449197860962567,
      "gold_tokens": 187,
      "predicted_tokens": 176,
      "scored_gold_speaker_tokens": 187,
      "scored_predicted_speaker_tokens": 176,
      "matched_scored_token_weight": 158.0,
      "correct_speake

## 11. Save, reload, and reproducibility notes

The training cell saves the resolved config, adapter, trainer state, and log history; the preceding inference section also reloads the adapter locally. Reproducibility still depends on using the same model revision, profile, seed, split, and processor settings.
What to check in saved artifacts: `config.resolved.yaml`, adapter files under `weights/<experiment>/`, trainer state and scores under `results/<experiment>/`, the experiments CSV row, and TensorBoard logs under `logs/<experiment>/`.

## Glossary

**QLoRA:** fine-tuning a quantized base model with low-rank trainable adapters.  
**LoRA adapter:** a small set of trainable low-rank matrices applied to a frozen model.  
**max_pixels:** the maximum resized image pixel area passed to the vision processor.  
**max_length:** the maximum combined token sequence length used by the trainer.  
**Gradient accumulation:** adding gradients across several batches before one optimizer update.  
**Epoch:** one pass through the training split.  
**Validation split:** held-out labeled sequences used to check behavior during development.  
**Parse failure:** a model response that cannot be converted into the required JSON schema.